# Few-shot Example Prep

Builds `tdr-experiments/data/few_shot_examples.jsonl`: up to 2 candidate
sentences per (homograph, variant) pair -- 82 pairs total (2 variants for
38 homographs, 3 for `ika`/`kasama`) -- sampled from real diacritized
usage in `tdr-experiments/data/TDR_train/`, for later use as few-shot
prompt examples.

`homographs`, `normalize_word`, `HOMOGRAPH_PATTERN`, `strip_invisible`,
`read_text_clean`, `split_sentences` are copied verbatim from
`dataprep.ipynb` (not imported across notebooks).

This notebook exports `few_shot_examples.jsonl` with `"sentence": null`
placeholders wherever real usage is too sparse to sample from -- it needs
a manual editing pass afterward to fill those in before actual few-shot
prompting.

## Config

`TRAIN_DIR` resolves to `tdr-experiments/data/TDR_train/` regardless of
whether this notebook's kernel working directory is the project root or
`tdr-experiments/` itself.

In [ ]:
from pathlib import Path

_candidates = [
    Path("tdr-experiments") / "data" / "TDR_train",
    Path("data") / "TDR_train",
]
TRAIN_DIR = next((p for p in _candidates if p.is_dir()), None)
assert TRAIN_DIR is not None, f"TDR_train not found in any of: {_candidates}"

print(f"TRAIN_DIR: {TRAIN_DIR.resolve()}")

## Shared setup: homographs, normalization, alternation regex, sentence splitter

Copied verbatim from `dataprep.ipynb`.

In [ ]:
import re

import pandas as pd

homographs = {
    "kaya": ["kayâ", "káya"],
    "bilang": ["biláng", "bílang"],
    "nasa": ["nasà", "nása"],
    "lamang": ["lamáng", "lámang"],
    "kasama": ["kasamá", "kasáma", "kasamâ"],
    "bagong": ["bagóng", "bágong"],
    "buhay": ["buháy", "búhay"],
    "dating": ["datíng", "dáting"],
    "silang": ["siláng", "sílang"],
    "pangulong": ["pangulóng", "pangúlong"],
    "tayo": ["tayô", "táyo"],
    "gabi": ["gabí", "gábi"],
    "kabilang": ["kabílang", "kabilâng"],
    "taong": ["taóng", "táong"],
    "patuloy": ["patulóy", "patúloy"],
    "pamahalaan": ["pámahalaán", "pámahaláan"],
    "lalaki": ["laláki", "lalakí"],
    "tapos": ["tapós", "tápos"],
    "pala": ["pála", "palá"],
    "baka": ["bakâ", "báka"],
    "paano": ["paanó", "paáno"],
    "huling": ["húling", "hulíng"],
    "sakit": ["sakít", "sákit"],
    "panalo": ["panaló", "panálo"],
    "kaibigan": ["kaibigán", "kaibígan"],
    "sasakyan": ["sasakyán", "sasákyan"],
    "ika": ["ikà", "ikâ", "iká"],
    "madaling": ["madalîng", "madáling"],
    "galing": ["galíng", "gáling"],
    "kita": ["kitá", "kíta"],
    "gamit": ["gamít", "gámit"],
    "huli": ["húli", "hulí"],
    "bukas": ["bukás", "búkas"],
    "kilala": ["kilalá", "kilála"],
    "galit": ["galít", "gálit"],
    "takot": ["takót", "tákot"],
    "puno": ["punô", "punò"],
    "pagod": ["pagód", "págod"],
    "abala": ["abalá", "abála"],
    "butas": ["butás", "bútas"],
}

assert len(homographs) == 40
print(f"Loaded {len(homographs)} target homographs.")

_DIACRITIC_MAP = {}
for _base, _marked in [("a", "áàâ"), ("e", "éèê"), ("i", "íìî"), ("o", "óòô"), ("u", "úùû")]:
    for _ch in _marked:
        _DIACRITIC_MAP[_ch] = _base
        _DIACRITIC_MAP[_ch.upper()] = _base.upper()
_DIACRITIC_TABLE = str.maketrans(_DIACRITIC_MAP)


def normalize_word(word: str) -> str:
    """Strips diacritics and lowercases -- used ONLY for matching a token
    against the `homographs` dict keys. Original casing/diacritics are
    preserved separately wherever a token's actual text is recorded."""
    return word.translate(_DIACRITIC_TABLE).lower()


# All known surface forms per homograph -- its bare (undiacritized) form
# plus every diacritized variant -- combined into a single alternation
# regex, so each file only needs one finditer pass. Longest forms first so
# a shorter form (e.g. "ika") can't win the alternation over a longer one
# that contains it.
_ALL_FORMS = set()
for _h, _variants in homographs.items():
    _ALL_FORMS.add(_h)
    _ALL_FORMS.update(_variants)

HOMOGRAPH_PATTERN = re.compile(
    r"\b(" + "|".join(re.escape(f) for f in sorted(_ALL_FORMS, key=len, reverse=True)) + r")\b",
    re.IGNORECASE,
)

INVISIBLE_CHARS = "\xad\ufeff\u200b\u200c\u200d\u2060"
INVISIBLE_TABLE = str.maketrans("", "", INVISIBLE_CHARS)


def strip_invisible(text: str) -> str:
    return text.translate(INVISIBLE_TABLE)


def read_text_clean(path: Path) -> str:
    return strip_invisible(path.read_text(encoding="utf-8"))


SENTENCE_SPLIT_PATTERN = re.compile(r'[.!?]+|\n')


def split_sentences(text: str) -> list[str]:
    """Splits on sentence-ending punctuation OR newlines; returns
    non-empty, whitespace-stripped segments in order."""
    return [seg.strip() for seg in SENTENCE_SPLIT_PATTERN.split(text) if seg.strip()]

## Step 1: Extract (homograph, variant) -> candidate sentences from TDR_train

Walks every file, splits into sentences, and for each sentence keeps only
DIACRITIZED target-homograph matches (`token.lower() !=
normalize_word(token)`) -- bare matches are excluded since a few-shot
example should demonstrate real diacritized usage, not the no-op bare
case. Each qualifying occurrence records its own `Full_Sentence` (the
sentence exactly as it appears in `TDR_train`, unmodified) under its
`(Homograph, Variant)` key in `candidates_by_variant`.

In [ ]:
from collections import defaultdict

train_files = sorted(TRAIN_DIR.glob("*.txt"))

candidates_by_variant = defaultdict(list)
total_sentences = 0

for path in train_files:
    text = read_text_clean(path)
    sentences = split_sentences(text)
    total_sentences += len(sentences)

    for sent_idx, sentence in enumerate(sentences):
        for match in HOMOGRAPH_PATTERN.finditer(sentence):
            token = match.group(0)
            homograph = normalize_word(token)
            variant = token.lower()

            if variant == homograph:
                continue  # bare -- not eligible as a few-shot example

            candidates_by_variant[(homograph, variant)].append({
                "Filename": path.name,
                "Sentence_Index": sent_idx,
                "Full_Sentence": sentence,
            })

print(f"Files processed:   {len(train_files)}")
print(f"Sentences scanned: {total_sentences}")
print(f"Distinct (homograph, variant) pairs with >=1 candidate: {len(candidates_by_variant)}")

## Step 2-3: Sample 2 examples per (homograph, variant), build records

For every one of the 82 `(homograph, variant)` pairs (every entry in
every homograph's `homographs[h]` list):
- 2+ candidates -> randomly sample 2 (fixed seed 42), both `"sampled"`.
- exactly 1 candidate -> use it (`"sampled"`), plus ONE
  `"MISSING_NEEDS_MANUAL_ENTRY"` placeholder (`Sentence = None`) so the
  gap is explicit rather than silently short.
- 0 candidates -> TWO `"MISSING_NEEDS_MANUAL_ENTRY"` placeholders.

One shared `random.Random(42)` is reused across all 82 pairs in a fixed
iteration order (`homographs` dict order, then each homograph's own
variant-list order), so the sampling is fully reproducible.

In [ ]:
import random

RANDOM_SEED = 42
rng = random.Random(RANDOM_SEED)

fewshot_rows = []
missing_pairs = []  # (homograph, variant, n_missing, n_already_sampled)

for h, variants in homographs.items():
    for v in variants:
        candidates = candidates_by_variant.get((h, v), [])
        n = len(candidates)

        if n >= 2:
            for c in rng.sample(candidates, 2):
                fewshot_rows.append({
                    "Homograph": h, "Variant": v, "Sentence": c["Full_Sentence"], "Status": "sampled",
                })
        elif n == 1:
            fewshot_rows.append({
                "Homograph": h, "Variant": v, "Sentence": candidates[0]["Full_Sentence"], "Status": "sampled",
            })
            fewshot_rows.append({
                "Homograph": h, "Variant": v, "Sentence": None, "Status": "MISSING_NEEDS_MANUAL_ENTRY",
            })
            missing_pairs.append((h, v, 1, 1))
        else:
            fewshot_rows.append({
                "Homograph": h, "Variant": v, "Sentence": None, "Status": "MISSING_NEEDS_MANUAL_ENTRY",
            })
            fewshot_rows.append({
                "Homograph": h, "Variant": v, "Sentence": None, "Status": "MISSING_NEEDS_MANUAL_ENTRY",
            })
            missing_pairs.append((h, v, 2, 0))

df_fewshot = pd.DataFrame(fewshot_rows, columns=["Homograph", "Variant", "Sentence", "Status"])

n_pairs = sum(len(v) for v in homographs.values())
print(f"Total (homograph, variant) pairs: {n_pairs} (should be 82)")
print(f"Total few-shot records built:     {len(df_fewshot)}")

## Step 4: Export

`ensure_ascii=False` so diacritics are written as literal UTF-8
characters. Placeholder records are written as-is with `"sentence":
null`, not dropped.

In [ ]:
import json

_data_candidates = [
    Path("tdr-experiments") / "data",
    Path("data"),
]
DATA_DIR = next((p for p in _data_candidates if p.is_dir()), None)
assert DATA_DIR is not None, f"tdr-experiments/data not found in any of: {_data_candidates}"

FEWSHOT_JSONL = DATA_DIR / "few_shot_examples.jsonl"

with open(FEWSHOT_JSONL, "w", encoding="utf-8") as f:
    for _, row in df_fewshot.iterrows():
        record = {
            "homograph": row["Homograph"],
            "variant": row["Variant"],
            "sentence": row["Sentence"],
            "status": row["Status"],
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Wrote {len(df_fewshot)} record(s) to {FEWSHOT_JSONL.resolve()}")

## Step 5: Report

In [ ]:
reread_count = sum(1 for _ in open(FEWSHOT_JSONL, encoding="utf-8"))
print(f"Total records exported: {reread_count}")

status_counts = df_fewshot["Status"].value_counts()
print("\nStatus breakdown:")
for status, count in status_counts.items():
    print(f"  {status:28s} {count}")

print(f"\nFULL LIST of (Homograph, Variant) pairs with at least one missing slot ({len(missing_pairs)} pair(s)):")
if missing_pairs:
    for h, v, n_missing, n_sampled in missing_pairs:
        if n_sampled:
            print(f"  {h} / {v}: need {n_missing} example ({n_sampled} already sampled)")
        else:
            print(f"  {h} / {v}: need {n_missing} examples")
else:
    print("  None -- every (homograph, variant) pair had 2+ real candidates.")

print("\n3 example fully-sampled records:")
sampled_rows = df_fewshot.loc[df_fewshot["Status"] == "sampled"]
for _, row in sampled_rows.head(3).iterrows():
    print(f"\n[{row['Homograph']} / {row['Variant']}]")
    print(f"  Sentence: {row['Sentence']}")

## Step 6: Inject manually-written sentences for missing slots

Reloads `few_shot_examples.jsonl` fresh from disk (not the in-memory
`df_fewshot`) so any manual edits already made to the file directly (e.g.
swapped-out sampled examples) are preserved rather than overwritten.

For each `(Homograph, Variant)` key in `MANUAL_SENTENCES`, its sentences
are consumed IN ORDER to fill that pair's `MISSING_NEEDS_MANUAL_ENTRY`
records (in file order). `MANUAL_SENTENCES`' total sentence count should
match the 31 missing slots from the 17 incomplete pairs (14 pairs needing
2 + 3 pairs needing 1 = 31) -- verified below before anything is written.

In [ ]:
MANUAL_SENTENCES = {
    ("nasa", "nasà"): ["Wala naman akong ibang nasà sa búhay kundi ikaw lang."],
    ("kasama", "kasamâ"): ["Gaano ba kasamâ ang tingin mo sa akin?"],
    ("tayo", "tayô"): ["Kanina pa yan siya tayô nang tayô."],
    ("kasama", "kasamá"): [
        "Kakampi natin ang mga kasamá sa pagkamit ng tunay na reporma sa lupa.",
        "Palagi na lang naaagrabyado ang mga kasamá sa bukid.",
    ],
    ("bagong", "bagóng"): [
        "Nakalampas na yata ang bagóng sinasakyan nila.",
        "Wala sa bagóng ito ang mga gamit namin!",
    ],
    ("pangulong", "pangulóng"): [
        "May binutinting na naman siyang pangulóng sa mga kalapati niya.",
        "Ano ba ang mabisang pangulóng sa mga mababait na ito?",
    ],
    ("gabi", "gábi"): [
        "Sinigang sa gábi, iyon talaga ang masarap!",
        "Hindi ako kumakain ng gábi.",
    ],
    ("patuloy", "patulóy"): [
        "Sa lahat-lahat ng kamag-anak namin, sila lang talaga ang nag-patulóy sa amin noong umuwi kami.",
        "Puwede bang patulóy ng kwento mo?",
    ],
    ("pamahalaan", "pámahaláan"): [
        "Bakit ba ang mga nasa gobyerno, hindi kayang pámahaláan nang wasto ang bansa?",
        "Pámahaláan mo ang barangay nang buong katapatan.",
    ],
    ("lalaki", "lalakí"): [
        "Konting tiyaga lang, lalakí ka pa naman.",
        "Ang sabi ng manghuhula, lalakí raw ang utang ko ngayong taon!",
    ],
    ("pala", "pála"): [
        "Pahiram daw mo ng pála si Papa!",
        "Hindi ko rin alam kung paano nagkasya ang pála sa toolbox niya.",
    ],
    ("paano", "paanó"): [
        "Sige na, paanó naman nitong ano ko o.",
        "Iyan, paanó ka kasi nang paanó, natuluyan ka tuloy.",
    ],
    ("kaibigan", "kaibigán"): [
        "Friends-friends asus, baka kaibigán!",
        "College kami nagkakilala ni Misis, noon din kami nag-kaibigán.",
    ],
    ("sasakyan", "sasákyan"): [
        "Sasákyan kita sa lahat ng gusto mo.",
        "Iyan na yata yung taxi na sasákyan natin!",
    ],
    ("madaling", "madáling"): [
        "Hala, bakâ madáling sobra ka ng mga nagsisihanap sa iyo niyan!",
        "Ang gustong madáling talaga ng mga pulis ay yung nag-iistambay sa kalye.",
    ],
    ("sakit", "sákit"): [
        "Natutuhan ko sa pilosopiya na ang sákit ay pagtitiis alang-alang sa pangarap.",
        "Pagdamay sa kapwa ang sákit sa kabila ng sariling paghihirap.",
    ],
    ("panalo", "panaló"): [
        "Bakit naman mukha mo ang ginamit mong panaló ng bola?",
        "May dala siyang panaló ng bulaklak para daw siguradong siya na ang sunod na ikakasal.",
    ],
}

manual_sentence_total = sum(len(v) for v in MANUAL_SENTENCES.values())
print(f"MANUAL_SENTENCES: {len(MANUAL_SENTENCES)} pair(s), {manual_sentence_total} sentence(s) total")
print(f"Expected missing slots: {len(missing_pairs)} pair(s), "
      f"{sum(n_missing for _, _, n_missing, _ in missing_pairs)} slot(s) total")
assert manual_sentence_total == sum(n_missing for _, _, n_missing, _ in missing_pairs), \
    "MANUAL_SENTENCES total doesn't match the number of missing slots"

with open(FEWSHOT_JSONL, encoding="utf-8") as f:
    fewshot_records = [json.loads(line) for line in f]

manual_queues = {key: list(sentences) for key, sentences in MANUAL_SENTENCES.items()}

filled_count = 0
for record in fewshot_records:
    if record["status"] != "MISSING_NEEDS_MANUAL_ENTRY":
        continue
    key = (record["homograph"], record["variant"])
    queue = manual_queues.get(key)
    if not queue:
        continue
    record["sentence"] = queue.pop(0)
    record["status"] = "manual_entry"
    filled_count += 1

print(f"\nFilled {filled_count} record(s) from MANUAL_SENTENCES.")
leftover = {key: sentences for key, sentences in manual_queues.items() if sentences}
if leftover:
    print(f"WARNING: unused manual sentence(s) left over: {leftover}")

## Step 7: Re-export and report

In [ ]:
from collections import Counter

with open(FEWSHOT_JSONL, "w", encoding="utf-8") as f:
    for record in fewshot_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Wrote {len(fewshot_records)} record(s) to {FEWSHOT_JSONL.resolve()}")

status_counts = Counter(r["status"] for r in fewshot_records)
print("\nStatus breakdown:")
for status, count in status_counts.items():
    print(f"  {status:15s} {count}")

still_missing = [r for r in fewshot_records if r["status"] == "MISSING_NEEDS_MANUAL_ENTRY"]
if still_missing:
    print(f"\nERROR: {len(still_missing)} record(s) still MISSING_NEEDS_MANUAL_ENTRY:")
    for r in still_missing:
        print(f"  {r['homograph']} / {r['variant']}")
else:
    print("\nZero records remain MISSING_NEEDS_MANUAL_ENTRY -- confirmed.")

print("\n5 example manual_entry records:")
manual_rows = [r for r in fewshot_records if r["status"] == "manual_entry"]
for r in manual_rows[:5]:
    print(f"\n[{r['homograph']} / {r['variant']}]")
    print(f"  Sentence: {r['sentence']}")